## Cell 1: Importing Libraries

This cell imports all the necessary Python libraries required for the project:

| Library | Purpose |
|---|---|
| `numpy` | Numerical computing, array operations |
| `pandas` | Data loading, manipulation, and analysis |
| `requests` | HTTP requests (imported but not used) |
| `pickle` | Saving/loading the trained model to disk |
| `re` | Regular expressions for text cleaning |
| `scipy.sparse` | Handling sparse matrices (imported but not used directly) |
| `train_test_split` | Splitting dataset into train and test sets |
| `LogisticRegression` | The ML classification algorithm |
| `TfidfVectorizer` | Converts text to numerical TF-IDF features |
| `PorterStemmer` | Reduces words to their root/stem form |
| `stopwords` | List of common English words to ignore |
| `accuracy_score` | Evaluates model performance |
| `nltk.download` | Downloads required NLTK language data files |

In [1]:
import numpy as np
import pandas as pd
import requests
import pickle
import re
import scipy.sparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Cell 2: Viewing English Stopwords

This cell prints all English stopwords from the NLTK library.

**Stopwords** are common words like *"a", "the", "is", "and"* that carry no meaningful sentiment information. Removing them reduces noise and improves model performance.

> **Example:** The sentence *"I am not happy"* after stopword removal becomes *"happy"* — only the meaningful word remains.

In [2]:
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

DATA Processing

## Cell 3: Loading the Dataset

This cell loads the Twitter Sentiment140 dataset from a CSV file.

- **`column_names`**: Manually assigned since the CSV has no header row.
- **`encoding='ISO-8859-1'`**: Required because the raw data contains non-UTF-8 characters (special characters in tweets).

The 6 columns are:

| Column | Description |
|---|---|
| `target` | Sentiment label: **0** = Negative, **4** = Positive |
| `id` | Unique tweet ID |
| `date` | Timestamp of the tweet |
| `flag` | Query used (always `NO_QUERY`) |
| `user` | Twitter username |
| `text` | The actual tweet content |

In [3]:
column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv(
    'twitter_data.csv',
    names=column_names,
    encoding='ISO-8859-1'
)

## Cell 4: Dataset Shape

Checks the **dimensions** of the loaded dataset.

- Output: `(1600000, 6)` — **1.6 million tweets** with **6 columns**.
- This is a large-scale dataset suitable for training a robust sentiment classifier.

In [4]:
# checking the number of rows and columns
twitter_data.shape

(1600000, 6)

## Cell 5: Preview the First 5 Rows

Displays the **first 5 rows** of the dataset using `.head()`.

This helps verify:
- Data loaded correctly
- Column names are properly assigned
- Tweet text looks as expected

In [5]:
#print 5 rows of dataset
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [6]:
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [7]:
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


## Cell 6: Checking for Missing Values

Uses `.isnull().sum()` to count missing (NaN) values in each column.

- **Output shows all zeros** — no missing values detected.
- This confirms the dataset is complete and ready for preprocessing.

In [8]:
# counting the number of missing valuse in the dataset
twitter_data.isnull().sum()

target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64

## Cell 7: Target Column Distribution

Checks how many tweets belong to each sentiment class.

- **Output:** 800,000 Negative (`0`) and 800,000 Positive (`4`)
- The dataset is **perfectly balanced** — equal samples for both classes.
- A balanced dataset prevents the model from being biased toward one class.

In [9]:
# checking the distribution of target column
twitter_data['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

convert target 4 to 1


## Cell 8: Converting Target Labels

Replaces the positive label from **4** to **1** using `.replace()`.

**Why?**
- Standard binary classification uses labels **0** and **1**.
- Label `4` is non-standard and could cause confusion.
- After this step: `0 = Negative`, `1 = Positive`

In [10]:
twitter_data.replace({'target': {4: 1}}, inplace=True)

## Cell 9: Verify Label Conversion

Verifies that the label replacement worked correctly.

- **Output:** 800,000 rows with label `0` and 800,000 with label `1`
- Confirms the dataset now uses standard binary labels.

In [11]:
# Checking the distribution of target column
twitter_data['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

**Stemming**

Stemming is the process of reducing a word to ist Root word

## Cell 10: Initialize Porter Stemmer

Creates an instance of **PorterStemmer** from NLTK.

The Porter Stemmer reduces words to their base/root form:
- "running" → "run"
- "happily" → "happi"
- "studies" → "studi"

This reduces vocabulary size and groups similar words together, improving model generalization.

In [12]:
post_stem=PorterStemmer()

## Cell 11: Defining the Stemming Function

Defines a custom `stemming()` function that preprocesses each tweet through these steps:

1. **NaN check** — Returns empty string `''` if content is a float (NaN value)
2. **Remove non-alphabetic characters** — `re.sub('[^a-zA-Z]', ' ', content)` strips numbers, punctuation, URLs, @mentions, hashtags
3. **Lowercase + split** — Converts to lowercase and splits into individual words
4. **Stopword removal + stemming** — Removes common English stopwords, then applies Porter stemming to each remaining word
5. **Rejoin** — Returns the cleaned words as a single string joined by spaces

> **Example:** `"@John I was running happily!"` → `"john run happili"`

In [13]:
stop_words = set(stopwords.words('english'))

def stemming(content):
    if isinstance(content, float):
        return ''
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower().split()
    stemmed_content = [post_stem.stem(word) for word in stemmed_content if word not in stop_words]
    return ' '.join(stemmed_content)

## Cell 12: Apply Stemming to All Tweets

Applies the `stemming()` function to every tweet in the `text` column and stores results in a new column called **`stemmed_content`**.

- Uses `.apply()` to run the function row-by-row.
- Creates the cleaned, preprocessed text used for model training.
- Processing 1.6 million tweets takes several minutes.

In [14]:
twitter_data['stemmed_content']=twitter_data['text'].apply(stemming)

## Cell 13: Preview Stemmed Data

Displays the first 5 rows again — now including the new **`stemmed_content`** column.

Comparison of original vs stemmed text:
- **Original:** `"is upset that he can't update his Facebook by texting..."`
- **Stemmed:** `"upset updat facebook text might cri result"` — stopwords removed, words reduced to roots

In [15]:
twitter_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


## Cell 14: Print All Stemmed Content

Prints the full `stemmed_content` column to verify stemming was applied across all 1.6 million rows.

Shows samples from the beginning (negative tweets) and end (positive tweets) of the dataset.

In [16]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


## Cell 15: Print Target Labels

Prints the `target` column to verify labels are correctly assigned:
- First rows: `0` (Negative)
- Last rows: `1` (Positive)

Confirms data integrity before splitting into features and labels.

In [17]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


## Cell 16: Separating Features and Labels

Splits the data into:
- **`X`** — Input features: the stemmed tweet text as a NumPy array
- **`Y`** — Output labels: `0` or `1` as a NumPy array

This separation is required for supervised machine learning — the model learns the mapping from X (text) → Y (sentiment).

In [18]:
# Seprating the data and label
X=twitter_data['stemmed_content'].values
Y=twitter_data['target'].values

## Cell 17: Preview Feature Array (X)

Prints the `X` array containing all preprocessed tweet texts.

- Each element is a clean, stemmed string ready for vectorization.
- Shape: `(1600000,)` — 1.6 million text samples.

In [19]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


## Cell 18: Preview Label Array (Y)

Prints the `Y` array containing all sentiment labels.

- Values are either `0` (Negative) or `1` (Positive).
- Shape: `(1600000,)` — matches the number of samples in X.

In [20]:
print(Y)

[0 0 0 ... 1 1 1]


Splitting the data into training data and test Data

## Cell 19: Train-Test Split

Splits the dataset into **training** and **testing** sets:

| Parameter | Value | Meaning |
|---|---|---|
| `test_size=0.2` | 20% | 320,000 tweets reserved for testing |
| `random_state=2` | Fixed seed | Ensures same split every run (reproducibility) |

- **Training set (X_train, y_train):** 1,280,000 tweets — used to train the model
- **Test set (X_test, y_test):** 320,000 tweets — used to evaluate on unseen data

In [21]:
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=2)

## Cell 20: Verify Split Shapes

Confirms the array sizes after splitting:

- `X.shape` = `(1600000,)` — Full dataset
- `X_train.shape` = `(1280000,)` — 80% for training
- `X_test.shape` = `(320000,)` — 20% for testing

In [22]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1280000,) (320000,)


## Cell 21: Preview Training Data

Prints a sample of the training set to verify the content looks correct — clean stemmed text strings ready for vectorization.

In [23]:
print(X_train)

['bad feel' 'love weekend' 'littlew bit far day trip fun get muddi today'
 ... 'twitter follow reform famou peopl'
 'aria cut hand open morn say like mommi anymor'
 'charlii sweet im excit aww miss u guy today']


## Cell 22: Preview Test Data

Prints a sample of the test set. This data is **never seen** by the model during training — used only for final evaluation to measure real-world performance.

In [24]:
print(X_test)

['brodiejay oh im go wow mona vale real place afteral know suck mvill slow train pffft'
 'babi grow' 'paint black roll stone best' ...
 'belladonna miss good music' 'reec r know imma get'
 'ooooooohnoooooo forgot charg ipod work one sad littl bar left']


## Cell 23: TF-IDF Vectorization

Converts text data into **numerical feature vectors** using TF-IDF (Term Frequency–Inverse Document Frequency):

```python
vectorizer = TfidfVectorizer()        # Create vectorizer
vectorizer.fit(X_train)               # Learn vocabulary from training data ONLY
X_train = vectorizer.transform(X_train)  # Transform training set
X_test = vectorizer.transform(X_test)    # Transform test set with same vocabulary
```

**Why TF-IDF?**
- Machine learning models cannot process raw text — they require numbers.
- TF-IDF assigns higher weights to words frequent in a tweet but rare overall (more informative).
- `.fit()` is called **only on training data** to prevent **data leakage**.

**Output:** Sparse matrix of shape `(1280000, ~461018)` — 1.28M tweets × ~461K unique word features.

In [25]:
#converting the textual data to numerical data
vectorizer=TfidfVectorizer()
vectorizer.fit(X_train)
X_train=vectorizer.transform(X_train)
X_test=vectorizer.transform(X_test)

## Cell 24: Inspect TF-IDF Matrix

Prints the TF-IDF sparse matrix structure.

- **Sparse matrix** stores only non-zero values to save memory (most tweets don't use most vocabulary words).
- Format: `(row_index, column_index) → tfidf_weight`
- The high number of stored elements (~9.4 million) reflects the vocabulary coverage across all training tweets.

In [26]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9452454 stored elements and shape (1280000, 461018)>
  Coords	Values
  (0, 31233)	0.7519171067329923
  (0, 129956)	0.6592576617698773
  (1, 241525)	0.5942540519072339
  (1, 437910)	0.8042773910733999
  (2, 42838)	0.2294043092907454
  (2, 93639)	0.16612263205577085
  (2, 128288)	0.2817039475788181
  (2, 140234)	0.2237237868613854
  (2, 145727)	0.16444046079173227
  (2, 237072)	0.6367700241935805
  (2, 278606)	0.48531397707654644
  (2, 411086)	0.18464287904169108
  (2, 416219)	0.2970321685021027
  (3, 8560)	0.37668420201866026
  (3, 179097)	0.4788299006944713
  (3, 201928)	0.524207488360723
  (3, 286683)	0.3682809811482607
  (3, 308910)	0.41292329558269175
  (3, 445747)	0.2188627839232472
  (4, 227660)	0.3291530007938817
  (4, 302738)	0.35966531233864635
  (4, 315347)	0.3291134109458034
  (4, 415818)	0.27525076285252903
  (4, 457632)	0.7604081439946883
  (5, 93639)	0.181548748438646
  :	:
  (1279996, 443184)	0.2023842381266662

Traning the ML MODEL

## Cell 25: Initialize Logistic Regression Model

Creates a **Logistic Regression** classifier:

- `max_iter=1000` — Allows up to 1000 iterations for convergence (default 100 is insufficient for large datasets).

**Why Logistic Regression for text classification?**
- Fast and memory-efficient on sparse TF-IDF matrices
- Outputs probability scores (used for confidence display)
- Works well for high-dimensional feature spaces
- Easy to interpret — word weights show which words drive sentiment

In [27]:
model=LogisticRegression(max_iter=1000)

## Cell 26: Train the Model

Trains the Logistic Regression model on the training data.

- **Input X_train:** TF-IDF feature matrix (shape: 1,280,000 × 461,018)
- **Input y_train:** Sentiment labels (0 or 1)
- The model learns which words are most associated with positive vs. negative sentiment.
- Training on 1.28 million examples may take a few minutes.

In [28]:
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Cell 27: Training Accuracy

Evaluates the model's performance on the **training data**.

- **Output:** ~80.24% accuracy
- The model correctly predicts sentiment for approximately 80% of the tweets it was trained on.
- This is the **in-sample** accuracy — expected to be higher than test accuracy.

In [29]:
# Accuracy on training data
X_train_pred = model.predict(X_train)
training_data_accuracy = accuracy_score(y_train, X_train_pred)
print('Accuracy score on the training data :', training_data_accuracy)

Accuracy score on the training data : 0.8023921875


## Cell 28: Saving the Model and Vectorizer

Saves both the trained model and the TF-IDF vectorizer to disk using **pickle**:

| File | Contents |
|---|---|
| `trained_model.sav` | Logistic Regression model weights |
| `vectorizer.sav` | Fitted TF-IDF vocabulary and IDF scores |

**Why save both?**
Any new tweet must be transformed using the **same vocabulary** learned during training. Without saving the vectorizer, feature dimensions would not match the model's expectations.

In [30]:
# Saving the trained model
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))

# Saving the vectorizer
pickle.dump(vectorizer, open('vectorizer.sav', 'wb'))

## Cell 29: Test Accuracy

Evaluates the model on **unseen test data** — the true measure of generalization.

- **Output:** ~77.66% accuracy
- The model correctly predicts sentiment for ~77.7% of tweets it has never seen before.
- The small gap between training (80.2%) and test (77.7%) accuracy indicates the model generalizes well without significant overfitting.

In [31]:
# Accuracy on test data
X_test_pred = model.predict(X_test)
test_data_accuracy = accuracy_score(y_test, X_test_pred)
print('Accuracy score on the test data :', test_data_accuracy)

Accuracy score on the test data : 0.77656875
